The purpose of this project is to analyze weekly supplement sales data to uncover trends, identify high-performing products and platforms, and evaluate the impact of discounts and returns. The analysis will help stakeholders make data-driven decisions to optimize sales and marketing strategies. 

SMART questions:
1. Which product categories generate the highest revenue weekly?
2. What is the average revenue per unit sold across all platforms?
3. Can optimizing discount rates improve units sold without significantly reducing revenue?
4. What platform consistently drives the most units sold, and should be prioritized in marketing efforts?


 | Activity | Description  |
 |--------|------------|
Data Cleaning|	Remove duplicates, check for nulls, fix date formats, and standardize values
Exploratory Data Analysis (EDA)|	Perform statistical analysis to identify patterns in revenue, units sold, returns, and discounts.
Visualization|	Develop charts showing revenue by category, top-selling products, discount impact, and location performance.
Insights & Recommendations|	Summarize key findings and propose actionable strategies for increasing future sales.


In [1]:
# Loading libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

In [2]:
# Loading the dataset
sales_data = pd.read_excel(r"D:\Project101\Food Supplement\dirty_supplement_sales_data_2020.xlsx")
sales_data.head()

,Date,Product Category,Product Name,Units Sold,Units Returned,Price,Discount (%),Revenue,Platform,Location
0,2020-10-07,Sleep Aids,ImmunoPro,NaN,1.0,$10.99,105.0,0.000,Instore,Los Angeles
1,2020-09-16,Vitamins,MuscleMax,NaN,-1.0,NaN,0.0,NaN,Mob App,Houston
2,2020-11-11,Immunity Boosters,Slim F@st,15.0,-1.0,15,NaN,225.000,Online,Chicago
3,2020-11-04,Energy Drinks,PowerFuel,14.0,0.0,$10.99,25.0,115.395,Online,new york
4,2020-06-03,Vitamins,SuperV!ta,NaN,NaN,$10.99,50.0,0.000,Mobile App,new york


In [3]:
# Make a copy the dataset
df = sales_data.copy()

In [5]:
# Checking descriptive statistics, basic info and nulls
print(f"Descriptive Statistics: {df.describe()}")
# Checking for missing values
print(f"Missing values in each column:\n {df.isnull().sum()}")
print("=" * 40)
print(f"{df.info()}")

Descriptive Statistics:         Units Sold  Units Returned  Discount (%)      Revenue
count  1551.000000     2495.000000   2651.000000  2457.000000
mean     24.514507        0.478557     35.403621   101.290307
std      14.398598        1.120023     35.162792   159.300390
min       1.000000       -1.000000      0.000000   -37.500000
25%      11.500000       -1.000000     10.000000     0.000000
50%      24.000000        0.000000     25.000000     0.000000
75%      37.000000        1.000000     50.000000   180.000000
max      50.000000        2.000000    105.000000   735.000000
Missing values in each column:
 Date                   0
Product Category       0
Product Name           0
Units Sold          1549
Units Returned       605
Price                643
Discount (%)         449
Revenue              643
Platform               0
Location               0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3100 entries, 0 to 3099
Data columns (total 10 columns):
 #   Column     

## Data cleaning

In [6]:
# Get the shape of the data
df.shape


(3100, 10)

In [7]:
# Checking for duplicates
print(f"Number of duplicate rows before: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Number of duplicate rows after: {df.duplicated().sum()}")

Number of duplicate rows before: 100
Number of duplicate rows after: 0


**Cleaning Categorical columns**

In [8]:
# Converting data column from object to date format
df["Date"] = pd.to_datetime(df["Date"], format='mixed')

In [9]:
# Product Name
print(f"Intial Product Names in dataset : {df['Product Name'].unique()}") # Checking unique entries
df["Product Name"] = df["Product Name"].replace(['Slim F@st','SuperV!ta'],['SuperVita','SlimFast'])
print(f"Cleaned Product Names in dataset : {df['Product Name'].unique()}") # Confirming changes

Intial Product Names in dataset : ['ImmunoPro' 'MuscleMax' 'Slim F@st' 'PowerFuel' 'SuperV!ta' 'SuperVita'
 'DreamRest' 'SlimFast']
Cleaned Product Names in dataset : ['ImmunoPro' 'MuscleMax' 'SuperVita' 'PowerFuel' 'SlimFast' 'DreamRest']


In [10]:
# Checking Platform column
print(df["Platform"].unique())
# Cleaning Platform column
df['Platform'] = df['Platform'].replace(['In-Store','Mob App','onlien'],['Instore','Mobile App','Online'])
# Confirming changes
print(f"New column values: {df['Platform'].unique()}")


['Instore' 'Mob App' 'Online' 'Mobile App' 'In-Store' 'onlien']
New column values: ['Instore' 'Mobile App' 'Online']


In [11]:
# Cleaning Location column
print(f"Original Location values: {df['Location'].unique()}")
df['Location'] = df['Location'].replace(['L.A.', 'new york', 'Chi-Town'], ['Los Angeles','New York', 'Chicago'])
print(f"Cleaned Location values: {df['Location'].unique()}")

Original Location values: ['Los Angeles' 'Houston' 'Chicago' 'new york' 'Phoenix' 'New York'
 'Chi-Town' 'L.A.']
Cleaned Location values: ['Los Angeles' 'Houston' 'Chicago' 'New York' 'Phoenix']


**Cleaning numeric data**

In [12]:
df.head()

,Date,Product Category,Product Name,Units Sold,Units Returned,Price,Discount (%),Revenue,Platform,Location
0,2020-10-07,Sleep Aids,ImmunoPro,NaN,1.0,$10.99,105.0,0.000,Instore,Los Angeles
1,2020-09-16,Vitamins,MuscleMax,NaN,-1.0,NaN,0.0,NaN,Mobile App,Houston
2,2020-11-11,Immunity Boosters,SuperVita,15.0,-1.0,15,NaN,225.000,Online,Chicago
3,2020-11-04,Energy Drinks,PowerFuel,14.0,0.0,$10.99,25.0,115.395,Online,New York
4,2020-06-03,Vitamins,SlimFast,NaN,NaN,$10.99,50.0,0.000,Mobile App,New York


In [23]:
print(f"Null values in each column:\n{df.isnull().sum()}")
print(df.info())

Null values in each column:
Date                   0
Product Category       0
Product Name           0
Units Sold          1492
Units Returned       587
Price                613
Discount (%)         434
Revenue              613
Platform               0
Location               0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 0 to 2999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              3000 non-null   datetime64[ns]
 1   Product Category  3000 non-null   object        
 2   Product Name      3000 non-null   object        
 3   Units Sold        1508 non-null   float64       
 4   Units Returned    2413 non-null   float64       
 5   Price             2387 non-null   object        
 6   Discount (%)      2566 non-null   float64       
 7   Revenue           2387 non-null   float64       
 8   Platform          3000 non-null   object        
 9   Location

In [26]:
print(1/0)

ZeroDivisionError: division by zero

In [28]:
def fill_nulls():
    # Filling Units Sold: (Revenue / Price)
    if df[df['Price']!=0]:
        df['Units Sold'] = df['Units Sold'].fillna(df['Revenue'] / df['Price'])
fill_nulls()

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [29]:
# Filling Nulls in the revenue column with zero
df["Revenue"]=df["Revenue"].fillna(0)

In [30]:
# Filling Nulls in the Units returned with column with nulls 
df["Units Returned"]=df["Units Returned"].fillna(0)
# Strip the -ve sign from Units returned
df["Units Returned"] = df["Units Returned"].abs()

In [31]:
# Dropping the dollar signand the USD str
df["Price"] = df["Price"].astype(str).str.replace(r'[^0-9.]', '', regex=True)

#Convert the column to numerical
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")


In [34]:

for average_price in df.groupby("Product Name")["Price"].mean():
    df["Price"].fillna(average_price)
df.head()

,Date,Product Category,Product Name,Units Sold,Units Returned,Price,Discount (%),Revenue,Platform,Location
0,2020-10-07,Sleep Aids,ImmunoPro,0.0,1.0,10.99,105.0,0.000,Instore,Los Angeles
1,2020-09-16,Vitamins,MuscleMax,0.0,1.0,NaN,0.0,0.000,Mobile App,Houston
2,2020-11-11,Immunity Boosters,SuperVita,15.0,1.0,15.00,NaN,225.000,Online,Chicago
3,2020-11-04,Energy Drinks,PowerFuel,14.0,0.0,10.99,25.0,115.395,Online,New York
4,2020-06-03,Vitamins,SlimFast,0.0,0.0,10.99,50.0,0.000,Mobile App,New York


In [ ]:
df.head()

,Date,Product Category,Product Name,Units Sold,Units Returned,Price,Discount (%),Revenue,Platform,Location
0,2020-10-07,Sleep Aids,ImmunoPro,0.0,1.0,10.99,105.0,0.000,Instore,Los Angeles
1,2020-09-16,Vitamins,MuscleMax,0.0,1.0,NaN,0.0,0.000,Mobile App,Houston
2,2020-11-11,Immunity Boosters,SuperVita,15.0,1.0,15.00,NaN,225.000,Online,Chicago
3,2020-11-04,Energy Drinks,PowerFuel,14.0,0.0,10.99,25.0,115.395,Online,New York
4,2020-06-03,Vitamins,SlimFast,0.0,0.0,10.99,50.0,0.000,Mobile App,New York


In [35]:
# Filling null Price with  mean
df["Price"] = df["Price"].fillna(df.groupby(["Product Category", "Product Name"])["Price"].transform("mean"))

In [36]:
df.isnull().sum()

Date                  0
Product Category      0
Product Name          0
Units Sold            0
Units Returned        0
Price                 0
Discount (%)        434
Revenue               0
Platform              0
Location              0
dtype: int64

In [40]:
# Calculating Discount and Revenue based on available data
if df['Discount (%)'].isnull and df['Revenue'].isnull:
    df['Discount (%)'] = df['Discount (%)'].fillna(0)
    df['Revenue'] = df['Price']* df['Units Sold']
 
elif df['Discount (%)'].isnull and df['Revenue'].notnull():
    df['Total_revenue'] = df['Price'] * df['Units Sold']
    df['Discount'] = df['Total_revenue'] - df['Revenue']
    df['Discount (%)'] = (df['Discount']/df['Total_Revenue'])*100
 
else:
    df['Revenue'] = (df['Price'] * df['Units Sold']) - (df['Price'] * (df['Discount (%)']/100))
 
df.head()

,Date,Product Category,Product Name,Units Sold,Units Returned,Price,Discount (%),Revenue,Platform,Location
0,2020-10-07,Sleep Aids,ImmunoPro,0.0,1.0,10.990000,105.0,0.00,Instore,Los Angeles
1,2020-09-16,Vitamins,MuscleMax,0.0,1.0,12.652917,0.0,0.00,Mobile App,Houston
2,2020-11-11,Immunity Boosters,SuperVita,15.0,1.0,15.000000,0.0,225.00,Online,Chicago
3,2020-11-04,Energy Drinks,PowerFuel,14.0,0.0,10.990000,25.0,153.86,Online,New York
4,2020-06-03,Vitamins,SlimFast,0.0,0.0,10.990000,50.0,0.00,Mobile App,New York


In [38]:
df.isnull().sum()

Date                0
Product Category    0
Product Name        0
Units Sold          0
Units Returned      0
Price               0
Discount (%)        0
Revenue             0
Platform            0
Location            0
dtype: int64